In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation"
os.makedirs(project_path, exist_ok=True)

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.6 MB/s eta 0:00:00


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-pose.pt")  # lightweight
# Other options:
# yolov8s-pose.pt
# yolov8m-pose.pt
# yolov8l-pose.pt

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# results = model.predict(
#     source="your_image.jpg",
#     conf=0.25,
#     device=0,
#     save=True,
#     project="/content/drive/MyDrive/yolo_pose_outputs",
#     name="image_test"
# )

results = model.predict(
    source=f"{project_path}/image.jpg",
    conf=0.25,
    device=0,
    save=True,
    project=project_path,
    name="yolo_image_predictions"
)


image 1/1 /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/image.jpg: 448x640 1 person, 7.5ms
Speed: 2.4ms preprocess, 7.5ms inference, 1.5ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/yolo_image_predictions


In [ ]:
# uploaded = files.upload()

In [ ]:
# results = model.predict(
#     source="your_video.mp4",
#     conf=0.3,
#     device=0,
#     save=True,
#     project="/content/drive/MyDrive/yolo_pose_outputs",
#     name="video_test"
# )

results = model.predict(
    source=f"{project_path}/video.mp4",
    conf=0.3,
    device=0,
    save=True,
    project=project_path,
    name="yolo_video_predictions"
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/341) /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/video.mp4: 384x640 6 persons, 7.8ms
video 1/1 (frame 2/341) /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/video.mp4: 384x640 6 persons, 8.9ms
video 1/1 (frame 3/341) /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/video.mp4: 384x640 6 persons, 11.2ms
video 1/1 (fr

In [ ]:
results = model.predict(f"{project_path}/image.jpg", device=0)

for r in results:
    keypoints = r.keypoints

    if keypoints is not None:
        print("Number of persons detected:", len(keypoints.xy))

        for i, person in enumerate(keypoints.xy):
            print(f"\nPerson {i+1}")
            print("Keypoints (x,y):")
            print(person)

            print("Confidence per keypoint:")
            print(keypoints.conf[i])


image 1/1 /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/image.jpg: 448x640 1 person, 10.6ms
Speed: 2.6ms preprocess, 10.6ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)
Number of persons detected: 1

Person 1
Keypoints (x,y):
tensor([[165.2007,  54.6166],
        [171.5421,  47.9308],
        [157.4192,  49.4908],
        [181.6473,  53.2805],
        [147.6667,  56.4289],
        [198.8831,  94.6246],
        [138.0496, 103.9973],
        [216.5442, 145.6991],
        [139.3189, 157.3458],
        [232.4578, 189.7829],
        [183.1963, 154.5241],
        [201.0831, 200.5470],
        [161.8345, 206.6545],
        [195.0915, 274.8596],
        [157.6969, 284.2261],
        [183.1552, 343.7513],
        [160.5061, 349.3743]], device='cuda:0')
Confidence per keypoint:
tensor([0.9885, 0.9553, 0.9638, 0.7833, 0.8375, 0.9976, 0.9979, 0.9865, 0.9907, 0.9756, 0.9833, 0.9994, 0.9995, 0.9981, 0.9985, 0.9833, 0.9858], device='cuda:

In [ ]:
# 0  nose
# 1  left_eye
# 2  right_eye
# 3  left_ear
# 4  right_ear
# 5  left_shoulder
# 6  right_shoulder
# 7  left_elbow
# 8  right_elbow
# 9  left_wrist
# 10 right_wrist
# 11 left_hip
# 12 right_hip
# 13 left_knee
# 14 right_knee
# 15 left_ankle
# 16 right_ankle

In [ ]:
import torch

def calculate_angle_torch(a, b, c):
    ba = a - b
    bc = c - b

    cosine_angle = torch.dot(ba, bc) / (torch.norm(ba) * torch.norm(bc))
    angle = torch.rad2deg(torch.acos(cosine_angle))

    return angle

results = model.predict(f"{project_path}/image.jpg", device=0)

for r in results:
    if r.keypoints is not None:
        kp = r.keypoints.xy[0]  # stays on GPU

        angle = calculate_angle_torch(
            kp[5],  # left_shoulder
            kp[7],  # left_elbow
            kp[9]   # left_wrist
        )

        print("Left elbow angle:", angle.item())


image 1/1 /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/image.jpg: 448x640 1 person, 15.4ms
Speed: 8.1ms preprocess, 15.4ms inference, 4.7ms postprocess per image at shape (1, 3, 448, 640)
Left elbow angle: 179.2259521484375


In [ ]:
# model = YOLO("yolov8n-pose.pt")

# model.train(
#     data="coco-pose.yaml",
#     epochs=100,
#     imgsz=640,
#     batch=16,
#     device=0,
#     project="/content/drive/MyDrive/yolo_pose_training",
#     name="pose_exp1"
# )

In [ ]:
# metrics = model.val()

# print("Pose mAP50:", metrics.pose.map50)
# print("Pose mAP50-95:", metrics.pose.map)

In [ ]:
model.track(
    source=f"{project_path}/video.mp4",
    device=0,
    save=True,
    project=project_path,
    name="yolo_video_tracking"
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/341) /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/video.mp4: 384x640 6 persons, 7.3ms
video 1/1 (frame 2/341) /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/video.mp4: 384x640 6 persons, 10.2ms
video 1/1 (frame 3/341) /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/pose estimation/video.mp4: 384x640 6 persons, 12.4ms
video 1/1 (f

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: ultralytics.engine.results.Keypoints object
 masks: None
 names: {0: 'person'}
 obb: None
 orig_img: array([[[128, 128, 128],
         [128, 128, 128],
         [128, 128, 128],
         ...,
         [148, 151, 149],
         [148, 151, 149],
         [148, 151, 149]],
 
        [[128, 128, 128],
         [128, 128, 128],
         [128, 128, 128],
         ...,
         [148, 151, 149],
         [148, 151, 149],
         [148, 151, 149]],
 
        [[128, 128, 128],
         [128, 128, 128],
         [128, 128, 128],
         ...,
         [148, 151, 149],
         [148, 151, 149],
         [148, 151, 149]],
 
        ...,
 
        [[200, 203, 201],
         [200, 203, 201],
         [200, 203, 201],
         ...,
         [229, 229, 229],
         [229, 229, 229],
         [229, 229, 229]],
 
        [[200, 203, 201],
         [200, 203, 201],
         [200, 203,